# UNVERIFIED / INCOMPLETE NOTEBOOK

**未検証・途中までの参考Notebookです。** 全セルの実行成功は確認していません。

**Stopping reason:** Automatic publication/registration stopped: [Errno 2] No such file or directory: '/home/phyto/phenotyping-colab/runs/p-50e1c2fb18dfbb4e76ee141d512cf54b-20261003T053520Z-b64d8905/queue-result.json'
直近のツールエラー（最終原因との一致はログで確認）:
[2026-10-03T05:40:29.055Z] ! permission requested: external_directory (/home/notes/*); auto-rejecting

Attempt status: failed. No manual login validation was performed. This draft may require your own authorized access, fail partway, or contain incomplete code. This attempt does not establish that the paper cannot be reproduced.

Paper: Evaluation of a low-cost staining method for improved visualization of sweet potato whitefly (Bemisia tabaci) eggs on multiple crop plant species

[PhenoPaper record](https://phenopaper.smartbreed-plant-phenotyping-platform.com/papers/p-50e1c2fb18dfbb4e76ee141d512cf54b)


# Reproduction: low-cost staining for *Bemisia tabaci* egg visualization

Reproduction of: **Van Raalte B, Prado J, Taddeo S, Mauck J (2024). "Evaluation of a low-cost staining method for improved visualization of sweet potato whitefly (*Bemisia tabaci*) eggs on multiple crop plant species." _Plant Methods_ 20:75.** DOI [10.1186/s13007-024-01209-z](https://doi.org/10.1186/s13007-024-01209-z) (CC BY 4.0).

**Scope (partial demonstration).** The paper stains/clears leaf discs and compares paired egg counts (two counters, before vs. after staining) in five crops. This notebook reproduces the **statistical core on the archived count data for Melon, Cowpea, and Tomato** — the crops covered by the authors' archived R script — by running the authors' own R code nearly verbatim and replicating their paired Wilcoxon signed-rank tests. The archived Zenodo record (11217925) does **not** include the separate cassava and sweet-potato analysis scripts that the paper's RMarkdown sources, so those two crops are out of scope here (this matches the paper's own code archive).

**Runtime:** CPU only — pure R statistics on small CSVs (<20 KB). No GPU is used or needed. Select **Runtime → Change runtime type → CPU**.

**日本語:** このノートブックは、論文の統計的中核（染色前後のペア卵数データの解析）を著者自身の R スクリプトで再現します。メロン・ササゲ・トマトの3作物が対象（アーカイブにカッサバ・サツマイモ用スクリプトは未収録）。CPU のみで実行できます。

| Paper operation (Rmd/script) | Notebook cell |
| --- | --- |
| Load Dryad count CSVs (`read.csv`) | Cell 3: download + SHA-256 verify |
| `combining_crops_stainunstain_analysis.R` (Zenodo 11217925) | Cell 5: run verbatim (paths only) |
| Paired Wilcoxon, `rstatix::wilcox_test(paired=TRUE)` | Cell 7: per crop per counter |
| Figures 2/S1-style count & % difference plots | Cell 8: display saved PNGs |

**Provenance:** author code = Zenodo record 11217925 (MIT license); data = Dryad doi:10.5061/dryad.vmcvdnd1m version 297485 (fetched at runtime from the Dryad API). Only edits to the authors' script: the six hard-coded local `read.csv` paths now point to `/content/data/`, and an appended driver saves/prints figures (`Rscript` does not auto-print ggplot objects). No scientific parameter is changed.

## 1. Setup: R runtime and packages

**Purpose / expected result.** The Colab Python runtime ships `Rscript`; we use Python only as a launcher for the authors' R analysis (per the verified CLI path for R code). This cell checks the R version and installs the exact packages the authors' script and tests use: `tidyr`, `dplyr`, `ggplot2`, `ggtext`, `ggsignif`, plus `rstatix` for the paired Wilcoxon tests. Expected: R ≥ 4.x and quiet package installs (~3–6 min on first run).

**日本語:** R 本体は Colab に同梱の `Rscript` を使用し、著者が使ったパッケージ(tidyr, dplyr, ggplot2, ggtext, ggsignif, rstatix)をインストールします。初回は数分かかります。

In [ ]:
import subprocess, textwrap
r = subprocess.run(["/usr/bin/Rscript", "-e",
    'cat(R.version.string, "\n"); ip <- rownames(installed.packages()); '
    'need <- setdiff(c("tidyr","dplyr","ggplot2","ggtext","ggsignif","rstatix"), ip); '
    'cat("missing:", paste(need, collapse=", "), "\n"); '
    'if (length(need)) install.packages(need, repos="https://cloud.r-project.org", quiet=TRUE); '
    'cat("installed:", paste(intersect(c("tidyr","dplyr","ggplot2","ggtext","ggsignif","rstatix"), rownames(installed.packages())), collapse=", "), "\n")'],
    capture_output=True, text=True, timeout=2400)
print(r.stdout[-2000:])
assert r.returncode == 0, r.stderr[-2000:]

## 2. Data: download the archived egg-count CSVs (Dryad)

**Purpose / expected result.** Download the seven egg-count CSVs used by the authors' script from the Dryad dataset **doi:10.5061/dryad.vmcvdnd1m, version 297485** via the documented public version-files API, and verify each file's SHA-256 against the digests published in the version file listing. Expected: 7 files, each matching its recorded digest; then a quick column/row summary. (The `wf_checks_*` and `README.md` files belong to other parts of the study and are not needed for this reproduction.)

**日本語:** 論文のカウント CSV 7件を Dryad 公開 API からダウンロードし、公開されている SHA-256 ダイジェストで検証します。

In [ ]:
import json, urllib.request, hashlib, csv, io, os

os.makedirs("/content/data", exist_ok=True)
VER = "297485"
with urllib.request.urlopen(
        f"https://datadryad.org/api/v2/versions/{VER}/files", timeout=120) as r:
    listing = json.load(r)
files = {f["path"]: f for f in listing["_embedded"]["stash:files"]}
NEEDED = [
    "egg_counts_instant_pot_test1.csv",
    "egg_counts_instant_pot_test2.csv",
    "egg_counts_stain_unstain_cowpea_rep4.csv",
    "egg_counts_stain_unstain_tgr_rep4-59do.csv",
    "egg_counts_stainunstain_tomato_rep2.csv",
    "egg_re_count_tomato_rep2pt2.csv",
]
for name in NEEDED:
    meta = files[name]
    url = "https://datadryad.org" + meta["_links"]["stash:download"]["href"]
    with urllib.request.urlopen(url, timeout=180) as r:
        blob = r.read()
    digest = hashlib.sha256(blob).hexdigest()
    assert digest == meta["digest"], (name, digest, meta["digest"])
    with open(f"/content/data/{name}", "wb") as f:
        f.write(blob)
    rows = list(csv.reader(io.StringIO(blob.decode("utf-8"))))
    print(f"{name}: {meta['size']} B, sha256 OK, {len(rows)-1} data rows, cols={rows[0]}")
print("All", len(NEEDED), "CSVs verified against Dryad version", VER)

# The author's script (Zenodo record 11217925) is embedded base64-encoded below to
# avoid any quoting/escaping corruption. It is decoded and written verbatim (with the
# documented path + driver edits) to /content, then executed with Rscript.
import base64, subprocess
r_b64 = "..."
open("/content/author_script.R", "wb").write(base64.b64decode(r_b64))
print("author_script.R bytes:", len(base64.b64decode(r_b64)))
r = subprocess.run(["/usr/bin/Rscript", "/content/author_script.R"],
                   capture_output=True, text=True, timeout=1800)
print(r.stdout)
assert r.returncode == 0, r.stderr[-3000:]
if r.stderr.strip(): print("STDERR (tail):", r.stderr[-1500:])

In [ ]:
R_CODE_PLACEHOLDER

# The Wilcoxon test script is likewise embedded base64-encoded and run with Rscript.
import base64, subprocess
w_b64 = "CiMjIFBhaXJlZCB0d28tc2lkZWQgV2lsY294b24gc2lnbmVkLXJhbmsgdGVzdHMgcGVyIGNyb3AgcGVyIGNvdW50ZXIsCiMjIHJlcGxpY2F0aW5nIHJlc3VsdHNfb3V0cHV0c192NC5SbWQgKGxpbmVzIH4xMDE0LTExODApOiByc3RhdGl4Ojp3aWxjb3hfdGVzdCguLi4sIHBhaXJlZCA9IFRSVUUpCiMjIG9uIHRoZSAqX2NvbXBsZXRlX3N0YWluX3dpZGUgZnJhbWVzIGJ1aWx0IGJ5IHRoZSBhdXRob3IncyBzY3JpcHQuCiMjIFNjb3BlIG5vdGU6IHRoZSBhcmNoaXZlZCBaZW5vZG8gcmVjb3JkIGxhY2tzIHRoZSBjYXNzYXZhL3N3ZWV0LXBvdGF0byBhbmFseXNpcwojIyBzY3JpcHRzLCBzbyBvbmx5IE1lbG9uLCBDb3dwZWEsIFRvbWF0byBhcmUgdGVzdGVkIGhlcmUuCgpsaWJyYXJ5KHJzdGF0aXgpCgpydW5fY3JvcCA8LSBmdW5jdGlvbihkZiwgc3BlY2llcykgewogIGxhcHBseShjKCJlZ2dfY291bnRfb25lIiwgImVnZ19jb3VudF90d28iLCAiY291bnRfcGVyZGlmZiIpLAogICAgICAgICBmdW5jdGlvbih2KSB7CiAgICAgICAgICAgcnN0YXRpeDo6d2lsY294X3Rlc3QoYXMuZm9ybXVsYShwYXN0ZSh2LCAifiBzdGFpbmVkIikpLCBkYXRhID0gZGYsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgcGFpcmVkID0gVFJVRSkgJT4lCiAgICAgICAgICAgICBtdXRhdGUoc3BlY2llcyA9IHNwZWNpZXMsIHZhcmlhYmxlID0gdiwKICAgICAgICAgICAgICAgICAgICBwX3ZhbHVlID0gcm91bmQocCwgNiksIFZfc3RhdGlzdGljID0gc3RhdGlzdGljKQogICAgICAgICB9KQp9CgpyZXMgPC0gYmluZF9yb3dzKAogIHJ1bl9jcm9wKHN1YnNldChtZWxvbl9jb21wbGV0ZV9zdGFpbl93aWRlLCAgc3BlY2llcyA9PSAiTWVsb24iKSwgICJNZWxvbiIpLAogIHJ1bl9jcm9wKHN1YnNldChjb3dwZWFfY29tcGxldGVfc3RhaW5fd2lkZSwgc3BlY2llcyA9PSAiQ293cGVhIiksICJDb3dwZWEiKSwKICBydW5fY3JvcChzdWJzZXQodG9tYXRvX2NvbXBsZXRlX3N0YWluX3dpZGUsIHNwZWNpZXMgPT0gIlRvbWF0byIpLCAiVG9tYXRvIikpCgpyZXMgJT4lIHNlbGVjdChzcGVjaWVzLCB2YXJpYWJsZSwgVl9zdGF0aXN0aWMsIHBfdmFsdWUpICU+JSBhcy5kYXRhLmZyYW1lKCkKCndyaXRlLmNzdihyZXMgJT4lIHNlbGVjdChzcGVjaWVzLCB2YXJpYWJsZSwgVl9zdGF0aXN0aWMsIHBfdmFsdWUpLAogICAgICAgICAgIi9jb250ZW50L2ZpZ3VyZXMvd2lsY294b25fcmVzdWx0cy5jc3YiLCByb3cubmFtZXMgPSBGQUxTRSkK"
open("/content/wilcoxon_tests.R", "wb").write(base64.b64decode(w_b64))
r = subprocess.run(["/usr/bin/Rscript", "/content/wilcoxon_tests.R"],
                   capture_output=True, text=True, timeout=1800)
print(r.stdout)
assert r.returncode == 0, r.stderr[-3000:]
import pandas as pd
pd.read_csv("/content/figures/wilcoxon_results.csv")

In [ ]:
WILCOX_PLACEHOLDER

## 5. Figures: paired counts and between-counter percent difference

**Purpose / expected result.** Displays the PNG figures saved by the authors' script: paired before→after egg counts per disc by counter (paper Figure 2–style), and boxplots of the between-counter percent difference before vs. after staining (paper Figure 3–style). Expected: after staining, counts shift upward and percent differences collapse toward 0 — visually matching the paper's figures.

**日本語:** 保存した図を表示します。染色後に卵数が増加し、カウンター間の相対差が0に近づくことが論文の図と同様に見えるはずです。

In [ ]:
from IPython.display import Image, display
for name in ["melon_counts_by_counter.png", "cowpea_counts_by_counter.png",
             "tomato_counts_by_counter.png", "melon_perdiff.png",
             "cowpea_perdiff.png", "tomato_perdiff.png"]:
    print(name)
    display(Image(filename=f"/content/figures/{name}", width=640))

## 6. Results vs. paper and limitations

**Expected comparison.** The Wilcoxon p-values/V statistics (cell 4) and the summary tables (cell 3) are compared with the paper's written statements and Tables S2–S3. Numerical results are deterministic (no randomness); only tiny deviations from the printed paper values are expected if the archived CSVs differ from the authors' working copies.

**Limitations.** (1) Three of five crops reproduced — the cassava and sweet-potato analysis scripts are not in the Zenodo archive. (2) Raw leaf-disc images are not deposited, so the image-based assessments are out of scope. (3) The authors' script edits paths only; all sample exclusions are kept as written.

**日本語:** 5作物のうち3作物を再現(アーカイブに残っていない2作物分のスクリプトのため)。結果は論文の Table S2/S3 および本文記載の統計量と比較してください。